In [1]:
from apiflask import Schema
from apiflask.fields import *
from flexiconc import Concordance

In [2]:
c = Concordance()
c.retrieve_from_cwb(corpus_name="GERMAPARL_1949_2021", query="[lemma='Test']")

INFO:ccc.cache:retrieving object "df_dump:fb4332e25b" from cache
INFO:ccc.cwb:using cached version "df_dump:fb4332e25b" of df_dump with 2544 matches
INFO:ccc.cwb:NQR "df_dbb0e62834" exists, overwriting
INFO:ccc.cqp:defining NQR "df_dbb0e62834" from dump with 2544 matches
INFO:ccc.cqp:saving NQR "GERMAPARL_1949_2021:df_dbb0e62834" to disk
INFO:ccc.cwb:assigned NQR "df_dbb0e62834" from dataframe
INFO:ccc.concordances:lines: selecting matches
INFO:ccc.concordances:lines: retrieving 2544 concordance line(s)
INFO:ccc.cwb:s-att "plenary_protocol" exists at 2544 of 2544 matches
INFO:ccc.cwb:s-att "lp" exists at 2544 of 2544 matches
INFO:ccc.cwb:s-att "protocol_no" exists at 2544 of 2544 matches
INFO:ccc.cwb:s-att "date" exists at 2544 of 2544 matches
INFO:ccc.cwb:s-att "year" exists at 2544 of 2544 matches
INFO:ccc.cwb:s-att "url" exists at 2544 of 2544 matches
INFO:ccc.cwb:s-att "filetype" exists at 2544 of 2544 matches
INFO:ccc.cwb:s-att "speaker_node" exists at 2544 of 2544 matches
INFO:cc

In [3]:
class AlgorithmOut(Schema):
    algorithm_name = String(required=True)
    args = Dict(required=True)

class AlgorithmsOut(Schema):
    ordering = Nested(AlgorithmOut, many=True, metadata={'nullable': True})
    grouping = Nested(AlgorithmOut, many=True, metadata={'nullable': True})


class TreeNodeOut(Schema):
    id = Integer(required=True)
    label = String(required=True)
    node_type = String(required=True)
    bookmarked = Boolean(required=True)
    children = Nested(lambda: TreeNodeOut(), many=True)
    line_count = Integer(metadata={'nullable': True})
    algorithms = Nested(AlgorithmsOut, metadata={'nullable': True})

In [4]:
TreeNodeOut().dump(c.root)

{'id': 0,
 'label': 'root',
 'node_type': 'subset',
 'bookmarked': False,
 'children': [{'id': 1,
   'label': 'Arrangement node',
   'node_type': 'arrangement',
   'bookmarked': False,
   'children': [],
   'algorithms': {'ordering': [{'algorithm_name': 'Sort by Corpus Position',
      'args': {}}],
    'grouping': None}}],
 'line_count': 2544}

In [37]:
class ArgOut(Schema):
    type = String()
    description = String()
    default = Raw()

class ArgsSchemaOut(Schema):
    required = List(String)
    properties = Dict(String, Nested(ArgOut))

class AlgorithmDetailsOut(Schema):
    full_name = String()
    algorithm_type = String()
    function = String()
    scope = String()
    args_schema = Nested(ArgsSchemaOut)

In [10]:
algos = c.find_node_by_id(1).available_algorithms()
algos = list(algos.values())

In [23]:
algos[0]['args_schema']['properties']

{'metadata_attribute': {'type': 'string',
  'description': "The metadata attribute to partition by (e.g., 'pos', 'speaker')."},
 'sort_by_partition_size': {'type': 'boolean',
  'description': 'If True, partitions will be sorted by size in descending order.',
  'default': True},
 'sorted_values': {'type': ['array', 'null'],
  'items': {'type': ['string', 'integer']},
  'description': 'If provided, partitions will be sorted by these specific values.'}}

In [38]:
AlgorithmDetailsOut().dump(algos[0])

{'full_name': 'Partition by Metadata Attribute',
 'algorithm_type': 'partitioning',
 'function': 'partition_by_metadata_attribute',
 'scope': None,
 'args_schema': {'required': ['metadata_attribute'],
  'properties': {'metadata_attribute': {'type': 'string',
    'description': "The metadata attribute to partition by (e.g., 'pos', 'speaker')."},
   'sort_by_partition_size': {'type': 'boolean',
    'description': 'If True, partitions will be sorted by size in descending order.',
    'default': True},
   'sorted_values': {'type': "['array', 'null']",
    'description': 'If provided, partitions will be sorted by these specific values.'}}}}

In [42]:
node = c.find_node_by_id(1)
[AlgorithmDetailsOut().dump(a) for a in list(node.available_algorithms().values())]

[{'full_name': 'Partition by Metadata Attribute',
  'algorithm_type': 'partitioning',
  'function': 'partition_by_metadata_attribute',
  'scope': None,
  'args_schema': {'required': ['metadata_attribute'],
   'properties': {'metadata_attribute': {'type': 'string',
     'description': "The metadata attribute to partition by (e.g., 'pos', 'speaker')."},
    'sort_by_partition_size': {'type': 'boolean',
     'description': 'If True, partitions will be sorted by size in descending order.',
     'default': True},
    'sorted_values': {'type': "['array', 'null']",
     'description': 'If provided, partitions will be sorted by these specific values.'}}}},
 {'full_name': 'Partition by Vectors',
  'algorithm_type': 'partitioning',
  'function': 'partition_by_vectors',
  'scope': None,
  'args_schema': {'required': ['vectors_column'],
   'properties': {'vectors_column': {'type': 'string',
     'description': 'The metadata column containing embeddings for each line.'},
    'n_partitions': {'type'